# WOP vs WoS (C++ only, unit cube)

Notebook compares three C++ configurations via existing `wop_cli`:
- `wop_escape`
- `wop_project`
- `wos`

Two views are reported:
- fixed-N comparison
- time-to-eps comparison


In [1]:
from __future__ import annotations

import json
import os
import statistics
import subprocess
import time
from pathlib import Path

import pandas as pd


def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root with external_wop_cpp")


WORKSPACE_ROOT = find_workspace_root(Path.cwd())
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"


def find_cpp_cli() -> Path:
    env_cli = os.environ.get("WOP_CPP_CLI")
    if env_cli:
        p = Path(env_cli).expanduser().resolve()
        if p.exists():
            return p

    candidates = [
        CPP_ROOT / "build" / "wop_cli",
        CPP_ROOT / "build" / "wop_cli.exe",
        CPP_ROOT / "build" / "Release" / "wop_cli.exe",
        CPP_ROOT / "build" / "Debug" / "wop_cli.exe",
    ]
    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(
        "wop_cli not found. Build external_wop_cpp first or set WOP_CPP_CLI env var."
    )


CPP_CLI = find_cpp_cli()
print(f"Using CLI: {CPP_CLI}")


Using CLI: c:\Users\Master\Desktop\Data\Диплом\external_wop_cpp\build\Release\wop_cli.exe


In [2]:
X0 = (3.0, 0.0, 0.0)
N_VALUES = [100, 1000, 10000, 100000, 1000000]
SEEDS = [314159, 271828, 161803]
MAX_STEPS = 200000
WARMUP_RUNS = 1
REPEATS = 3
EPS_TARGETS = [2e-1, 1e-1, 5e-2, 2e-2]

CONFIGS = {
    "wop_escape": {
        "method": "wop",
        "args": ["--r-max", "1000000", "--r-max-mode", "escape", "--r-max-factor", "3.0"],
    },
    "wop_project": {
        "method": "wop",
        "args": ["--r-max", "0", "--r-max-mode", "project", "--r-max-factor", "3.0"],
    },
    "wos": {
        "method": "wos",
        "args": ["--delta", "1e-3", "--rho-scale", "1.0", "--rho1-scale", "2.0"],
    },
}


In [3]:
def run_cpp_once(config_name: str, n_paths: int, seed: int) -> dict:
    cfg = CONFIGS[config_name]
    cmd = [
        str(CPP_CLI),
        "--example", "box",
        "--method", str(cfg["method"]),
        "--x0", f"{X0[0]} {X0[1]} {X0[2]}",
        "--n", str(int(n_paths)),
        "--seed", str(int(seed)),
        "--max-steps", str(int(MAX_STEPS)),
        "--json",
        *[str(x) for x in cfg["args"]],
    ]

    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0

    payload = json.loads(out)
    payload["config"] = config_name
    payload["wall_time_sec"] = float(elapsed)
    return payload


def run_with_repeats(config_name: str, n_paths: int, seed: int) -> dict:
    for _ in range(WARMUP_RUNS):
        _ = run_cpp_once(config_name=config_name, n_paths=n_paths, seed=seed)

    runs = [run_cpp_once(config_name=config_name, n_paths=n_paths, seed=seed) for _ in range(REPEATS)]
    time_values = [float(r["wall_time_sec"]) for r in runs]

    ref = dict(runs[0])
    ref["wall_time_sec"] = float(statistics.median(time_values))
    return ref


In [4]:
rows = []
for config_name in CONFIGS:
    for n_paths in N_VALUES:
        for seed in SEEDS:
            row = run_with_repeats(config_name=config_name, n_paths=n_paths, seed=seed)
            row["seed"] = int(seed)
            rows.append(row)
            print(
                f"config={config_name:>11} n={n_paths:>7} seed={seed} "
                f"J={float(row['J']):.8f} abs_error={float(row['abs_error']):.3e} "
                f"eps={float(row['eps']):.3e} time={float(row['wall_time_sec']):.3f}s"
            )

df_raw = pd.DataFrame(rows)
df_raw


config= wop_escape n=    100 seed=314159 J=0.33829499 abs_error=1.659e-02 eps=1.217e-01 time=0.006s
config= wop_escape n=    100 seed=271828 J=0.39224553 abs_error=3.736e-02 eps=1.308e-01 time=0.006s
config= wop_escape n=    100 seed=161803 J=0.40722724 abs_error=5.234e-02 eps=1.356e-01 time=0.006s
config= wop_escape n=   1000 seed=314159 J=0.35509678 abs_error=2.101e-04 eps=3.945e-02 time=0.010s
config= wop_escape n=   1000 seed=271828 J=0.36599346 abs_error=1.111e-02 eps=4.020e-02 time=0.010s
config= wop_escape n=   1000 seed=161803 J=0.37850080 abs_error=2.361e-02 eps=4.041e-02 time=0.011s
config= wop_escape n=  10000 seed=314159 J=0.35304366 abs_error=1.843e-03 eps=1.259e-02 time=0.062s
config= wop_escape n=  10000 seed=271828 J=0.36108122 abs_error=6.195e-03 eps=1.262e-02 time=0.061s
config= wop_escape n=  10000 seed=161803 J=0.36014924 abs_error=5.263e-03 eps=1.264e-02 time=0.062s
config= wop_escape n= 100000 seed=314159 J=0.35460767 abs_error=2.791e-04 eps=3.985e-03 time=0.777s


,method,x0,n_total,n_truncated,J,exact,abs_error,S2,eps,mean_steps,config,wall_time_sec,seed
0,wop,"[3, 0, 0]",100,57,0.338295,0.354887,0.016592,0.164627,0.121723,39.260000,wop_escape,0.006249,314159
1,wop,"[3, 0, 0]",100,53,0.392246,0.354887,0.037359,0.190048,0.130784,37.910000,wop_escape,0.005751,271828
2,wop,"[3, 0, 0]",100,53,0.407227,0.354887,0.052341,0.204224,0.135573,37.330000,wop_escape,0.005685,161803
3,wop,"[3, 0, 0]",1000,559,0.355097,0.354887,0.000210,0.172902,0.039448,38.949000,wop_escape,0.010228,314159
4,wop,"[3, 0, 0]",1000,552,0.365993,0.354887,0.011107,0.179556,0.040200,39.129000,wop_escape,0.010455,271828
5,wop,"[3, 0, 0]",1000,537,0.378501,0.354887,0.023614,0.181435,0.040409,39.668000,wop_escape,0.011041,161803
6,wop,"[3, 0, 0]",10000,5653,0.353044,0.354887,0.001843,0.176024,0.012587,41.246200,wop_escape,0.061519,314159
7,wop,"[3, 0, 0]",10000,5553,0.361081,0.354887,0.006195,0.177049,0.012623,40.761400,wop_escape,0.060928,271828
8,wop,"[3, 0, 0]",10000,5570,0.360149,0.354887,0.005263,0.177450,0.012637,40.680400,wop_escape,0.061516,161803
9,wop,"[3, 0, 0]",100000,56372,0.354608,0.354887,0.000279,0.176405,0.003985,41.456130,wop_escape,0.777193,314159


In [5]:
fixed_n = (
    df_raw.groupby(["config", "n_total"], as_index=False)
    .agg(
        abs_error_median=("abs_error", "median"),
        eps_median=("eps", "median"),
        time_median_sec=("wall_time_sec", "median"),
        mean_steps_median=("mean_steps", "median"),
        trunc_rate=("n_truncated", lambda s: float(s.sum()) / float((df_raw.loc[s.index, "n_total"]).sum())),
    )
    .sort_values(["n_total", "config"])
    .reset_index(drop=True)
)

print("Fixed-N comparison")
fixed_n


Fixed-N comparison


,config,n_total,abs_error_median,eps_median,time_median_sec,mean_steps_median,trunc_rate
0,wop_escape,100,0.037359,0.130784,0.005751,37.910000,0.543333
1,wop_project,100,0.039285,0.119214,0.006161,22.690000,0.000000
2,wos,100,0.016435,0.077976,0.009047,29.260000,0.000000
3,wop_escape,1000,0.011107,0.040200,0.010455,39.129000,0.549333
4,wop_project,1000,0.011477,0.038443,0.013952,21.342000,0.000000
5,wos,1000,0.004408,0.025396,0.030676,30.183000,0.000000
6,wop_escape,10000,0.005263,0.012623,0.061516,40.761400,0.559200
7,wop_project,10000,0.002144,0.012108,0.060373,21.711700,0.000000
8,wos,10000,0.000959,0.008000,0.116803,29.960600,0.000000
9,wop_escape,100000,0.001420,0.003987,0.777193,41.456130,0.563010


In [6]:
records = []
for config_name, grp in fixed_n.groupby("config"):
    grp_sorted = grp.sort_values("n_total")
    for eps_target in EPS_TARGETS:
        ok = grp_sorted[grp_sorted["eps_median"] <= eps_target]
        if len(ok) == 0:
            records.append({
                "config": config_name,
                "eps_target": float(eps_target),
                "min_time_sec": float("nan"),
                "n_at_target": int(-1),
            })
            continue

        best = ok.iloc[ok["time_median_sec"].argmin()]
        records.append({
            "config": config_name,
            "eps_target": float(eps_target),
            "min_time_sec": float(best["time_median_sec"]),
            "n_at_target": int(best["n_total"]),
        })

time_to_eps = pd.DataFrame(records).sort_values(["eps_target", "config"]).reset_index(drop=True)
print("Time-to-eps comparison")
time_to_eps


Time-to-eps comparison


,config,eps_target,min_time_sec,n_at_target
0,wop_escape,0.02,0.061516,10000
1,wop_project,0.02,0.060373,10000
2,wos,0.02,0.116803,10000
3,wop_escape,0.05,0.010455,1000
4,wop_project,0.05,0.013952,1000
5,wos,0.05,0.030676,1000
6,wop_escape,0.10,0.010455,1000
7,wop_project,0.10,0.013952,1000
8,wos,0.10,0.009047,100
9,wop_escape,0.20,0.005751,100


In [7]:
artifacts_dir = WORKSPACE_ROOT / "tmp" / "jupyter-notebook" / "wop-vs-wos-cpp-unit-cube"
artifacts_dir.mkdir(parents=True, exist_ok=True)
(artifacts_dir / "raw_rows.json").write_text(df_raw.to_json(orient="records", indent=2), encoding="utf-8")
(artifacts_dir / "fixed_n.json").write_text(fixed_n.to_json(orient="records", indent=2), encoding="utf-8")
(artifacts_dir / "time_to_eps.json").write_text(time_to_eps.to_json(orient="records", indent=2), encoding="utf-8")
print("Saved artifacts to", artifacts_dir)


Saved artifacts to c:\Users\Master\Desktop\Data\Диплом\tmp\jupyter-notebook\wop-vs-wos-cpp-unit-cube
